In [1]:
# import packages
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path
from typing import Tuple, Union

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from catboost import CatBoostClassifier
from classifier_calibration.calibration_error import classwise_ece
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import  log_loss
from sklearn.metrics import precision_score, recall_score

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
np.random.seed(42)

In [4]:
# import custom functions
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Thesis code" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from Functions.data_utils import (
    plot_incremental_response_rate,
    uplift_by_decile_bin,
    coerce_metrics_to_numeric,
)

Failed to import duecredit due to No module named 'duecredit'


In [5]:
file_path = r"Data/df_preds.csv"
df_preds = pd.read_csv(file_path)

## Diagnostics calibration and performance per class

In [6]:
def metrics_per_model_per_class(df: pd.DataFrame, eps: float = 1e-15) -> pd.DataFrame:
    labels = (
        [f"reactivated_{i}" for i in range(8)] +
        [f"no_reactivated_{i}" for i in range(8)]
    )
    
    rows = []
    for model, group in df.groupby("model"):
        prec = precision_score(group["y_true"], group["y_pred"], labels=labels, average=None, zero_division=0)
        rec = recall_score(group["y_true"], group["y_pred"], labels=labels, average=None, zero_division=0)
        
        for i, label in enumerate(labels):
            prob_col = f"p_{label}"
            if prob_col in group.columns:
                mean_prob = group[prob_col].mean()
                std_prob = group[prob_col].std()
                
                # OvR log loss: binary log loss of class k vs rest
                y_true_bin = (group["y_true"] == label).astype(int)
                p_k = np.clip(group[prob_col].to_numpy(), eps, 1 - eps)
                class_logloss = float(
                    log_loss(
                        y_true_bin,
                        np.column_stack([1 - p_k, p_k]),
                        labels=[0, 1],
                    )
                )
            else:
                mean_prob = np.nan
                std_prob = np.nan
                class_logloss = np.nan
            
            rows.append({
                "model": model,
                "class": label,
                "precision": prec[i],
                "recall": rec[i],
                "mean_predicted_prob": mean_prob,
                "std_predicted_prob": std_prob,
                "class_logloss": class_logloss,
            })
    
    return pd.DataFrame(rows)
    
out_dir = Path("Output/classification_output")
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df = metrics_per_model_per_class(df_preds)
metrics_df.to_excel(out_dir / "metrics_per_model_per_class.xlsx", index=False)

In [7]:
# Classwise-ECE

proba_cols = [
    "p_reactivated_0",
    "p_reactivated_1",
    "p_reactivated_2",
    "p_reactivated_3",
    "p_reactivated_4",
    "p_reactivated_5",
    "p_reactivated_6",
    "p_reactivated_7",
    "p_no_reactivated_0",
    "p_no_reactivated_1",
    "p_no_reactivated_2",
    "p_no_reactivated_3",
    "p_no_reactivated_4",
    "p_no_reactivated_5",
    "p_no_reactivated_6",
    "p_no_reactivated_7"
]

class_mapping = {
    "reactivated_0": 0,
    "reactivated_1": 1,
    "reactivated_2": 2,
    "reactivated_3": 3,
    "reactivated_4": 4,
    "reactivated_5": 5,
    "reactivated_6": 6,
    "reactivated_7": 7,
    "no_reactivated_0": 8,
    "no_reactivated_1": 9,
    "no_reactivated_2": 10,
    "no_reactivated_3": 11,
    "no_reactivated_4": 12,
    "no_reactivated_5": 13,
    "no_reactivated_6": 14,
    "no_reactivated_7": 15,
}
def compute_classwise_ece_per_model(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model, g in df.groupby("model"):
        y_prob = g[proba_cols].to_numpy()
        y_true_int = g["y_true"].map(class_mapping).to_numpy()

        if np.isnan(y_true_int).any():
            bad_labels = (
                g.loc[np.isnan(y_true_int), "y_true"]
                .unique()
                .tolist()
            )
            raise ValueError(f"Unmapped y_true labels found for model {model}: {bad_labels}")

        y_true_int = y_true_int.astype(int)

        ece_per_class = classwise_ece(
            labels=y_true_int,
            pred_probs=y_prob
        )

        rows.append(
            {
                "model": model,
                "classwise_ece": float(np.mean(ece_per_class)),
            }
        )

    return pd.DataFrame(rows)


ece_df = compute_classwise_ece_per_model(df_preds)
ece_df

,model,classwise_ece
0,catboost_calibrated_dirichlet,0.057414
1,catboost_uncal,0.050089
2,rf_calibrated_dirichlet,0.054331
3,rf_uncal,0.025781


In [8]:
# Averaged log loss over all classes, making minority classes equal in weight
def logloss_per_model_class(
    df: pd.DataFrame,
    proba_cols: list[str] = proba_cols,
    class_mapping: dict[str, int] = class_mapping,
    eps: float = 1e-15,
) -> pd.DataFrame:
    class_labels = [c.replace("p_", "") for c in proba_cols]
    inv_class_mapping = {v: k for k, v in class_mapping.items()}

    rows: list[dict] = []

    for model, g in df.groupby("model"):
        y_true_int = g["y_true"].map(class_mapping).to_numpy()

        if np.isnan(y_true_int).any():
            bad_labels = (
                g.loc[np.isnan(y_true_int), "y_true"]
                .unique()
                .tolist()
            )
            raise ValueError(f"Unmapped y_true labels found for model {model}: {bad_labels}")

        y_true_int = y_true_int.astype(int)
        y_prob = g[proba_cols].to_numpy()

        model_logloss_values = []

        # Per-class metrics (one-vs-rest)
        for k in range(len(proba_cols)):
            y_true_bin = (y_true_int == k).astype(int)

            p_k = np.clip(y_prob[:, k], eps, 1 - eps)

            ll = float(
                log_loss(
                    y_true_bin,
                    np.column_stack([1 - p_k, p_k]),
                    labels=[0, 1],
                )
            )

            model_logloss_values.append(ll)

            rows.append(
                {
                    "model": model,
                    "class": inv_class_mapping[k],
                    "logloss": ll,
                }
            )

        # ---- Mean log loss row ----
        rows.append(
            {
                "model": model,
                "class": "mean",
                "logloss": float(np.mean(model_logloss_values)),
            }
        )

    return pd.DataFrame(rows)


metrics_df = logloss_per_model_class(df_preds)
metrics_df[metrics_df['class']=='mean']

,model,class,logloss
16,catboost_calibrated_dirichlet,mean,0.230645
33,catboost_uncal,mean,0.209259
50,rf_calibrated_dirichlet,mean,0.253959
67,rf_uncal,mean,0.272431


In [9]:
# Mutliclass logloss evaluates full probbaility vector of all classes, dominated by majority group 
def multiclass_logloss_per_model(
    df: pd.DataFrame,
    proba_cols: list[str] = proba_cols,
    class_mapping: dict[str, int] = class_mapping,
) -> pd.DataFrame:
    rows = []
    for model, g in df.groupby("model"):
        y_true = g["y_true"].map(class_mapping).to_numpy().astype(int)
        y_prob = g[proba_cols].to_numpy()
        ll = log_loss(y_true, y_prob)
        rows.append({"model": model, "multiclass_logloss": float(ll)})
    return pd.DataFrame(rows)

mc_ll_df = multiclass_logloss_per_model(df_preds)
mc_ll_df

,model,multiclass_logloss
0,catboost_calibrated_dirichlet,2.722859
1,catboost_uncal,2.392160
2,rf_calibrated_dirichlet,3.121065
3,rf_uncal,3.537670
